In [ ]:
import os 
import pandas as pd
def addToDataFrame(df, path, label):
    for f in os.listdir(path):
        if f.endswith('.png'):
            df = pd.concat([df, pd.DataFrame([{'path': os.path.join(path, f), 'label': label}])], ignore_index=True)
            
    return df
df = pd.DataFrame(columns=['path', 'label'])
df = addToDataFrame(df, 'Dataset_DICOM/Bleeding', '1') 
df = addToDataFrame(df, 'Dataset_DICOM/Ischemia', '2')
df = addToDataFrame(df, 'Dataset_DICOM/Normal', '0')

In [ ]:
from sklearn.model_selection import train_test_split
train, validation = train_test_split(df, test_size=0.2, random_state=42)
print(f'Training samples: {len(train)}')
print(f'Validation samples: {len(validation)}')


In [ ]:
import numpy as np
from PIL import Image

def extract_features(dataframe, model):
    features_list = []
    labels_list = []
    for idx, row in dataframe.iterrows():
        img = Image.open(row['path']).convert('RGB').resize((224, 224))
        img_array = np.array(img) / 255.0 
        img_array = np.expand_dims(img_array, axis=0)
        features = model.predict(img_array)
        features = features.reshape(features.shape[0], -1)
        features_list.append(features)
        labels_list.append(int(row['label']))
    
    X = np.vstack(features_list)
    y = np.array(labels_list)
    return X, y

X_train, y_train = extract_features(train, VGG_model)
X_val, y_val = extract_features(validation, VGG_model)